# 06.10 - CNNs for Computer Vision

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Convolutional Neural Networks use convolutional filters to detect spatial patterns (edges, textures, shapes) in grid-structured data like images. They are the foundation of computer vision.

## 2. Why Does This Matter?

Fully connected layers ignore spatial structure. CNNs exploit spatial locality and translation invariance, making them far more efficient and effective for images.

## 3. Prerequisites

- Units 06.4, 06.8 (PyTorch, training loops), basic linear algebra

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build and train CNNs on synthetic image-like tensors
- Explain convolution, pooling, feature maps
- Compute conv/pool output sizes

## 5. Mental Model

A CNN is a feature-detector hierarchy: early layers detect edges, middle layers combine them into patterns, late layers into object parts, then a classifier maps features to classes.

> NOT using torchvision (not installed / no pretrained downloads): we train small custom CNN models on synthetic 8x8 grayscale tensors.


## 6. Backend + Convolution Ideation


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

torch.manual_seed(42); np.random.seed(42)
print("PyTorch version:", torch.__version__)


## 7. Convolution from Scratch (1 filter)

Slide a 3x3 edge-detector kernel over a small image and count feature outputs.


In [ ]:
img = torch.randn(1, 1, 6, 6)
kernel = torch.tensor([[[[-1., 0, 1], [-1, 0, 1], [-1, 0, 1]]]])  # vertical edge
out = F.conv2d(img, kernel, padding=1)
print("Input shape:", img.shape)
print("Filter shape:", kernel.shape)
print("Output feature map shape:", out.shape)
print("\nA single 3x3 conv with padding keeps spatial size; a feature map per filter.")


## 8. CNN Output-Size Computation

Verify the size formula `out = (W - K + 2P)/S + 1` with PyTorch.


In [ ]:
def conv_out(W, K, P, S):
    return (W - K + 2 * P) // S + 1

W, K, P, S = 28, 3, 1, 1
print(f"Conv: W={W}, K={K}, P={P}, S={S} -> out={conv_out(W,K,P,S)} (expected 28)")
W2, P2, S2 = conv_out(28, 3, 1, 1), 0, 2
print(f"MaxPool2d(2): {conv_out(28,2,0,2)} -> after pool: {conv_out(W2,2,0,S2)} (expected 14)")

# Confirm with a live module
x = torch.randn(1, 1, 28, 28)
c = nn.Conv2d(1, 8, 3, padding=1)
p = nn.MaxPool2d(2)
print("Live: conv ->", c(x).shape, "-> pool ->", p(c(x)).shape)


## 9. Define a Small CNN

Convs + ReLU + pooling feeding a linear classifier, for 8x8 input.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8->4
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 4->2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

m = SimpleCNN()
print(f"Parameters: {sum(p.numel() for p in m.parameters()):,}")
print("Feature-map trace:", m.features(torch.randn(1, 1, 8, 8)).shape)


## 10. Train the CNN on Synthetic Spatially-Patterned Images

Classes = where the bright blob appears (top-left vs bottom-right).


In [ ]:
def synth_images(n, seed=0):
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, 1, 8, 8)) * 0.1
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        X[i, 0, 1:3, 1:3] += 1.0 if i % 2 == 0 else 0.0
        X[i, 0, 5:7, 5:7] += 1.0 if i % 2 == 1 else 0.0
        y[i] = 0 if i % 2 == 0 else 1
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

X, y = synth_images(800)
split = 600
Xtr, ytr = X[:split], y[:split]
Xte, yte = X[split:], y[split:]
print("Train:", Xtr.shape, "Test:", Xte.shape)

cnn = SimpleCNN()
crit = nn.CrossEntropyLoss()
opt = optim.Adam(cnn.parameters(), lr=0.003)
for epoch in range(25):
    opt.zero_grad()
    loss = crit(cnn(Xtr), ytr)
    loss.backward(); opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:2d}: loss={loss.item():.4f}")
with torch.no_grad():
    tr_acc = (cnn(Xtr).argmax(dim=1) == ytr).float().mean().item()
    te_acc = (cnn(Xte).argmax(dim=1) == yte).float().mean().item()
print(f"CNN train acc: {tr_acc:.3f}, test acc: {te_acc:.3f}")


## 11. Compare CNN vs MLP on the Same Images

Show the CNN sees spatial structure an MLP (flatten) struggles with.


In [ ]:
mlp = nn.Sequential(nn.Flatten(), nn.Linear(8*8, 64), nn.ReLU(), nn.Linear(64, 2))
opt = optim.Adam(mlp.parameters(), lr=0.003)
for _ in range(25):
    opt.zero_grad(); crit(mlp(Xtr), ytr).backward(); opt.step()
with torch.no_grad():
    mlp_acc = (mlp(Xte).argmax(dim=1) == yte).float().mean().item()
print(f"MLP  test acc: {mlp_acc:.3f}")
print(f"CNN  test acc: {te_acc:.3f}")
print("\nCNNs exploit local spatial structure; this problem's blob location is spatial.")


## 12. Debugging / Common Errors

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Shape mismatch | Wrong conv/pool output size | Compute sizes manually | Use padding='same' or fix Linear input |
| CNN worse than MLP | Too little data / wrong arch | Check dataset size | Transfer learning, more data |
| Very slow | Large input, no pooling | Check input dims | Add pooling, reduce size |
| Overfit small images | Model too large | Check param count | Augment, pretrained |

## 13. Real-World Considerations

- Self-driving cars use CNNs for lanes/pedestrians/signs (e.g., 30ms per 640x480 frame on GPU).
- Each filter yields one feature map; multiple filters give depth.

## 14. When NOT to Use

- Non-grid/tabular data — use an MLP.

## 15. Challenge

Make the blob task invariant to translation: place the blob at random positions and see whether pooling helps robustness.


In [ ]:
# Challenge: translation-invariance intuition
g = np.random.default_rng(3)
Xr = g.standard_normal((400, 1, 8, 8)) * 0.1
yr = np.zeros(400, dtype=np.int64)
for i in range(400):
    r0 = g.integers(0, 6); c0 = g.integers(0, 6)  # random location
    Xr[i, 0, r0:r0+2, c0:c0+2] += (1.0 if i % 2 == 0 else 0.0)
    yr[i] = 0 if i % 2 == 0 else 1
Xr = torch.tensor(Xr, dtype=torch.float32); yr = torch.tensor(yr, dtype=torch.long)
cnn2 = SimpleCNN(); opt = optim.Adam(cnn2.parameters(), lr=0.003)
for _ in range(25):
    opt.zero_grad(); crit(cnn2(Xr), yr).backward(); opt.step()
with torch.no_grad():
    acc = (cnn2(Xr).argmax(dim=1) == yr).float().mean().item()
print(f"CNN on translation-randomized blobs: train acc {acc:.3f}")
print("Convolution+pooling gives some translation tolerance by design.")


## 16. Closed-Book Recall

Without looking back:

1. What does a convolutional filter detect?
2. Why does pooling help translation invariance?
3. How do you compute a conv layer's output size?
4. Why are CNNs more parameter-efficient than MLPs for images?

## 17. Teach-Back Questions

Explain to another person:

- How a kernel slides and produces a feature map.
- Why early features differ from late features.

## 18. Summary

You implemented a conv op, verified sizes, trained a CNN on synthetic spatial images, and compared it to an MLP.

## 19. Further Experiment

- Try average pooling vs max pooling.
- Vary number of filters.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
